# Maximum Flow with the Ford-Fulkerson Algorithm

## Introduction
The **maximum flow problem** asks: given a directed network where each edge has a
capacity, what is the largest amount of "stuff" (water, data, traffic) that can be
sent from a **source** node `S` to a **sink** node `T` without exceeding any edge capacity?

The **Ford-Fulkerson method** solves this by repeatedly:
1. Finding an **augmenting path**: a path from `S` to `T` in the *residual graph*
   where every edge still has spare capacity.
2. Pushing flow equal to the **bottleneck** (the smallest residual capacity on that path).
3. Updating the residual graph: forward edges lose capacity, and reverse edges gain
   capacity (which lets later paths "undo" earlier decisions).

The process stops when no augmenting path exists. The total flow pushed is the maximum flow.

## What this notebook does
- Represents the network as an **adjacency matrix** (`graph[u][v]` = residual capacity of `u → v`)
- Finds augmenting paths using **Depth-First Search (DFS)**
- Computes the max flow of a classic 6-node network (the CLRS textbook example)
- Analyses the time and space complexity

## Step 1: Import Libraries
We use **NumPy** for two things:
- Storing the capacity matrix as a 2D integer array
- Creating the `visited` array (`np.full`) and the value `np.inf`, which acts as the
  starting "infinite" bottleneck

In [1]:
import numpy as np

## Step 2: DFS to Find an Augmenting Path

`dfs(graph, curr, t, vst, path)` searches for a path from `curr` to the sink `t`
that uses only edges with **positive residual capacity**.

| Parameter | Meaning |
|---|---|
| `graph` | Residual capacity matrix. `graph[u][v] > 0` means `u → v` can carry more flow |
| `curr` | Node currently being explored |
| `t` | Sink node |
| `vst` | Boolean array marking nodes already on the current path (prevents cycles) |
| `path` | List filled with the nodes of the path found (modified in place) |

**How it works**
1. Append `curr` to `path`. If `curr == t`, the sink is reached and the function returns `True`.
2. Mark `curr` as visited.
3. For every neighbour `i` with `graph[curr][i] > 0` that is not visited, recurse.
   If the recursive call succeeds, return `True` immediately.
4. If no neighbour leads to the sink, **backtrack**: unmark `curr`, remove it from `path`,
   and return `False`.

When it returns `True`, `path` holds the full route, for example `[0, 1, 2, 4, 3, 5]`.

> **Note:** Because `vst[curr]` is reset to `False` when backtracking, a node can be
> re-explored via a different route. 

In [2]:
def dfs(graph,curr,t,vst,path):
        path.append(curr)
        if(curr==t):return True
        vst[curr] = True
        for i in range(len(graph[curr])):
           if graph[curr][i]>0 and vst[i]==False:
                
                if(dfs(graph,i,t,vst,path)):return True
                
        vst[curr] = False
        path.pop()
        return False

## Step 3: The Ford-Fulkerson Main Loop

`ford_fulkerson(graph, s, t)` repeatedly calls `dfs` until no augmenting path remains.

**Algorithm**
1. Set `max_flow = 0`.
2. While `dfs` finds a path from `s` to `t`:
   - **Find the bottleneck**: `min_wt` is the minimum residual capacity along the path.
   - **Augment the flow**, for every edge `(u → v)` on the path:
     - `graph[u][v] -= min_wt` (less spare capacity forward)
     - `graph[v][u] += min_wt` (more capacity to cancel the flow backward)
   - Add `min_wt` to `max_flow` and reset `path`.
3. Return `max_flow`.

**Key idea:** the reverse-edge update is what makes the algorithm correct. It allows
later paths to reroute flow that an earlier path sent in a poor direction.

> The function **modifies `graph` in place**, so we pass `graph.copy()` when calling it
> to keep the original capacities intact.

In [3]:
def ford_fulkerson (graph,s,t):
    #s is source
    #t is target or sink
    n= len(graph)
    path =[]
    max_flow =0 
    while(dfs(graph,s,t,np.full(n,False),path)):
        min_wt= np.inf
        
        for i in range (len(path)-1):
            min_wt = min(graph[path[i]][path[i+1]],min_wt)
        
        for i in range (len(path)-1):
            graph[path[i]][path[i+1]] -= min_wt
            graph[path[i+1]][path[i]] += min_wt
        
        max_flow += min_wt
        path = []
    
    
    return max_flow    
        
    
        

## Step 4: Define the Flow Network

The network has 6 nodes, indexed as:

| Index | 0 | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|---|
| Name | S (source) | A | B | C | D | T (sink) |

Entry `graph[i][j]` is the capacity of the directed edge `i → j` (0 means no edge).

| Edge | Capacity | Edge | Capacity |
|---|---|---|---|
| S → A | 16 | B → D | 14 |
| S → B | 13 | C → B | 9 |
| A → B | 10 | C → T | 20 |
| A → C | 12 | D → C | 7 |
| B → A | 4 | D → T | 4 |

Note that `A → B` and `B → A` both exist, which is why the matrix is not symmetric.

In [4]:
graph = np.array([
    [0, 16, 13,  0,  0,  0],  # S
    [0,  0, 10, 12,  0,  0],  # A
    [0,  4,  0,  0, 14,  0],  # B
    [0,  0,  9,  0,  0, 20],  # C
    [0,  0,  0,  7,  0,  4],  # D
    [0,  0,  0,  0,  0,  0]   # T
], dtype=int)

## Step 5: Run the Algorithm

We call `ford_fulkerson` with:
- `graph.copy()`: a copy, so the original `graph` matrix is preserved
- `s = 0`: the source `S`
- `t = 5`: the sink `T`

The returned value is the maximum flow, which we print.

In [5]:
max_flow = ford_fulkerson(graph.copy(), 0, 5)
print(f"Max Flow: {max_flow}")

Max Flow: 23


## Results

```
Max Flow: 23
```

**The maximum flow from S to T is 23 units.**

### Augmenting paths found (in the order the DFS discovers them)

| # | Path | Bottleneck | Total flow |
|---|---|---|---|
| 1 | S → A → B → D → C → T | 7 | 7 |
| 2 | S → A → B → D → T | 3 | 10 |
| 3 | S → A → C → B → D → T | 1 | 11 |
| 4 | S → A → C → T | 5 | 16 |
| 5 | S → B → A → C → T | 6 | 22 |
| 6 | S → B → C → T | 1 | 23 |

The 7th DFS call finds no path, so the algorithm stops. Paths 5 and 6 use
**reverse (residual) edges**, which cancel part of the flow sent by earlier paths.

### Verification with the Max-Flow Min-Cut Theorem
In the final residual graph, the nodes still reachable from S are `{S, A, B, D}`.
The edges leaving this set are:

- A → C = 12
- D → C = 7
- D → T = 4

Their total capacity is 12 + 7 + 4 = **23**, matching the max flow.
So 23 is optimal.


## Time and Space Complexity

Let **V** = number of vertices, **E** = number of edges, and **F** = value of the maximum flow.

### Time Complexity

| Part | Cost |
|---|---|
| One DFS (adjacency matrix, each node scans a full row) | O(V²) |
| Bottleneck search and augmentation along a path | O(V) |
| Number of augmentations (integer capacities, each pushes ≥ 1 unit) | ≤ F |

**Overall (standard Ford-Fulkerson with DFS): O(F · V²)**

- The running time depends on the **flow value F**, not just the graph size.
  With large capacities this is *pseudo-polynomial*.
- For this example, V = 6 and F = 23, so the work is tiny (only 6 augmentations).
- **Caveat for this implementation:** `dfs` resets `vst[curr] = False` on backtracking.
  This lets it re-explore nodes through different routes, so a single DFS can enumerate
  many simple paths, and its worst case on dense graphs is **exponential in V**.
  Removing that line (the standard "mark once" DFS) restores the O(V²) bound per search
  without affecting correctness.
- With non-integer capacities, plain Ford-Fulkerson may fail to terminate.
- **Edmonds-Karp** (BFS instead of DFS) guarantees O(V · E²) regardless of capacities.

### Space Complexity

| Structure | Space |
|---|---|
| Residual capacity matrix (the copy of `graph`) | O(V²) |
| `vst` array | O(V) |
| `path` list and recursion stack (depth ≤ V) | O(V) |

**Overall: O(V²)**, dominated by the adjacency matrix.

## Conclusion
Ford-Fulkerson finds the maximum flow of this network, **23**, using six augmenting
paths. The result is confirmed by a minimum cut of the same capacity.